# World Cup Predictor 2026 — Exploração com Dados Reais

Este notebook demonstra como substituir os dados mock da V1 por dados reais
do dataset Kaggle de resultados internacionais.

**Pré-requisito**: baixe o CSV do Kaggle e salve em `data/raw/results.csv`:
```bash
pip install kaggle
kaggle datasets download martj42/international-football-results-from-1872-to-2017
unzip international-football-results-from-1872-to-2017.zip -d ../data/raw/
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_fetcher import load_kaggle_results, filter_copa_teams, COPA_2026_TEAMS
from data_loader import load_matches_kaggle
from elo_model import build_elo_history
from feature_engineering import build_team_strengths, create_elo_diff
from goal_model import estimate_lambdas_for_fixture, calculate_score_matrix, probabilities_from_matrix
from backtest import run_backtest, _print_report

## 1. Carregamento e inspeção dos dados reais

In [ ]:
KAGGLE_CSV = '../data/raw/results.csv'

matches = load_kaggle_results(KAGGLE_CSV, min_date='2000-01-01')
print(f'Partidas: {len(matches):,}')
print(f'Período : {matches.data_jogo.min().date()} → {matches.data_jogo.max().date()}')
print(f'Gols/time (média): {matches[["gols_time_a","gols_time_b"]].values.mean():.3f}')
matches.head()

In [ ]:
matches['competicao'].value_counts().plot(kind='barh', figsize=(8,5))
plt.title('Distribuição de competições (dados reais)')
plt.xlabel('Partidas')
plt.tight_layout()
plt.show()

## 2. Distribuição de gols: reais vs. mock

In [ ]:
import data_loader
mock = data_loader.generate_mock_matches(n_matches=600)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df, title in zip(axes, [matches, mock], ['Reais (Kaggle)', 'Mock V1']):
    gols = pd.concat([df['gols_time_a'], df['gols_time_b']])
    gols.value_counts().sort_index().plot(kind='bar', ax=ax)
    ax.set_title(f'Distribuição de gols — {title}')
    ax.set_xlabel('Gols por time por partida')
    ax.set_ylabel('Frequência')
plt.tight_layout()
plt.show()

## 3. Ratings Elo construídos a partir de dados reais

In [ ]:
ratings = build_elo_history(matches)
elo_df = pd.Series(ratings).sort_values(ascending=False)

# Top 20
elo_df.head(20).plot(kind='barh', figsize=(8, 6))
plt.gca().invert_yaxis()
plt.title('Top 20 seleções por Elo (dados reais, pós-2000)')
plt.xlabel('Elo')
plt.tight_layout()
plt.show()

## 4. Força de ataque/defesa das seleções da Copa 2026

In [ ]:
matches_copa = filter_copa_teams(matches)
matches_copa = create_elo_diff(matches_copa, ratings)
strengths = build_team_strengths(matches_copa, ratings, n_games=10)

strengths[['elo','forca_ofensiva','fragilidade_defensiva','saldo_medio_gols']] \
    .sort_values('elo', ascending=False).head(15)

## 5. Previsão de um confronto com dados reais

In [ ]:
time_a, time_b = 'Brazil', 'Germany'
la, lb = estimate_lambdas_for_fixture(
    time_a, time_b, strengths,
    diferenca_elo=ratings.get(time_a, 1500) - ratings.get(time_b, 1500)
)
matrix = calculate_score_matrix(la, lb)
probs = probabilities_from_matrix(matrix)

print(f'{time_a} x {time_b}')
print(f'  λ_A = {la:.3f}   λ_B = {lb:.3f}')
print(f'  Vitória {time_a}: {probs["prob_vitoria_time_a"]*100:.0f}%')
print(f'  Empate        : {probs["prob_empate"]*100:.0f}%')
print(f'  Vitória {time_b}: {probs["prob_vitoria_time_b"]*100:.0f}%')
print(f'  Over 2.5      : {probs["prob_over_2_5"]*100:.0f}%')
print(f'  Placar prov.  : {probs["placar_mais_provavel"]}')

## 6. Backtest walk-forward com dados reais

In [ ]:
# Usa os dados reais em vez dos mock — apenas troca o argumento `matches`
result = run_backtest(matches=matches_copa, warmup=200, n_games=10)
_print_report(result)

## 7. Curva de calibração (dados reais)

In [ ]:
from evaluation import calibration_curve_data, plot_calibration_curve

preds = result['predictions']
y_prob = preds[['prob_a','prob_empate','prob_b']].values
y_true = preds['resultado'].values  # 0=A, 1=empate, 2=B

cal_data = calibration_curve_data(y_true, y_prob)
plot_calibration_curve(cal_data)